1- Loading the data from kaggle

2- Load labels csv for labels that contain image ID and breed

3- Checking the breed count

4- One-hot encoding on labels dats prediction column

5- Load the images, convert them to an array and normalize them

6- Check the shape and size of x and y data

7- Building the model network architecture

8- Split the data and fit it into the model and create an accuracy plot

API token: KGAT_cf826c84137705f0eb2580e5be71c9e7

To use this token, set the KAGGLE_API_TOKEN environment variable: export KAGGLE_API_TOKEN=KGAT_cf826c84137705f0eb2580e5be71c9e7

After setting KAGGLE_API_TOKEN, you can use the client as follows: kaggle competitions list

In [ ]:
#from google.colab import files
#files.upload()

In [ ]:
# install the kaggle API client
!pip install -q kaggle

In [ ]:
# The Kaggle API client expects this file to be in ~/.kaggle, so move it there
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle.json
# This permissions change avoids a warning on Kaggle tool startup
!chmod 600 ~/.kaggle/kaggle.json

Setting up Kaggle using Kaggle API

In [ ]:
!mkdir dog_dataset
%cd dog_dataset

To store the data we will create a new directory and make it as current working directory

In [ ]:
!kaggle datasets list -s dogbreedidfromcomp

Searching Kaggle for the required dataset using search option(-s) with title 'dogbreedidfromcomp'. We can also use different search options like searching competitions, notebooks, kernels, datasets, etc

In [ ]:
# Downloading dataset and coming out of directory
!kaggle datasets download catherinehorng/dogbreedidfromcomp
%cd ..

Dataset URL: https://www.kaggle.com/datasets/catherinehorng/dogbreedidfromcomp
License(s): unknown
100% 691M/691M [00:09<00:00, 76.8MB/s]

/content


After searching the data next step would be downloading the data into collab notebook using references found in search option

In [ ]:
# Unzipping downloaded file and removing unusable file
!unzip dog_dataset/dogbreedidfromcomp.zip -d dog_dataset
!rm dog_dataset/dogbreedidfromcomp.zip
!rm dog_dataset/sample_submission.csv

In [ ]:
# Important library imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from keras.preprocessing import image
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from keras.optimizers import Adam

In [ ]:
labels_all = pd.read_csv('dog_dataset/labels.csv')
labels_all.head()

In [ ]:
breed_counts = labels_all['breed'].value_counts()
breed_counts.head()

In [ ]:
# selecting first 3 breeds due to computation power
selected = ['scottish_deerhound', 'maltese_dog', 'afghan_hound']
labels = labels_all[(labels_all['breed'].isin(selected))]
labels = labels.reset_index()
labels.head()

In [ ]:
x_data = np.zeros((len(labels), 224, 224, 3), dtype = 'float32')
y_data = label_binarize(labels['breed'], classes = selected)
for i in tqdm(range(len(labels))):
  img = image.load_img('dog_dataset/train/%s.jpg' % labels['id'][i], target_size=(224, 224))
  img = image.img_to_array(img)
  x = np.expand_dims(img.copy(), axis = 0)
  x_data[i] = x/255.0
print(f'Train images shape: {x_data.shape} Size: {x_data.size}')
print(f'One hot encoded output shape: {y_data.shape} Size: {y_data.size}')

In [ ]:
# Building the model

model = Sequential()

model.add(Conv2D(filters = 64, kernel_size = (5,5), activation ='relu', input_shape = (224,224,3)))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 32, kernel_size = (3,3), activation ='relu', kernel_regularizer = 'l2'))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 16, kernel_size = (7,7), activation ='relu', kernel_regularizer = 'l2'))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 8, kernel_size = (5,5), activation ='relu', kernel_regularizer = 'l2'))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Flatten())
model.add(Dense(128, activation = "relu", kernel_regularizer = 'l2'))
model.add(Dense(64, activation = "relu", kernel_regularizer = 'l2'))
model.add(Dense(len(selected), activation = "softmax"))

model.compile(loss = 'categorical_crossentropy', optimizer = Adam(0.0001),metrics=['accuracy'])

model.summary()

In [ ]:
# Splitting the data set into training and testing data sets
x_train_and_val, x_test, y_train_and_val, y_test = train_test_split(x_data, y_data, test_size = 0.1)
# Splitting the training data set into training and validation data sets
x_train, x_val, y_train, y_val = train_test_split(x_train_and_val, y_train_and_val, test_size = 0.1)

In [ ]:
x_test

In [ ]:
x_test[1,:,:,:]

In [ ]:
# training the model
epochs, batch_size = 100, 128
history = model.fit(x_train, y_train, batch_size = batch_size, epochs = epochs, validation_data = (x_val, y_val))

In [ ]:
# plot the training history

plt.figure(figsize = (12, 5))
plt.plot(history.history['accuracy'], color = 'r')
plt.plot(history.history['val_accuracy'], color='b')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend(['train', 'val'])

plt.show()

In [ ]:
y_pred = model.predict(x_test)
score = model.evaluate(x_test, y_test)
print('Accuracy over the test set: \n ', round((score[1]*100), 2), '%')

In [ ]:
# Plotting image to compare
plt.imshow(x_test[1,:,:,:])
plt.show()

# Finding max value from predition list and comaparing original value vs predicted
print("Originally : ",labels['breed'][np.argmax(y_test[1])])
print("Predicted : ",labels['breed'][np.argmax(y_pred[1])])

In [ ]:
model.save('dog_breed.h5')